# 🚀 Amazon India — Super Data Cleaning Notebook

| Step | Column | Source |
|------|--------|--------|
| 1 | Dates | data_cleaning_pipeline — regex + audit trail |
| 2 | Prices | Combined — NB1 diagnostics + NB3 regex map |
| 3 | Ratings | 01_data_cleaning — fractions + median fill |
| 4 | Cities | 01_data_cleaning — 13 mappings |
| 5 | Booleans | data_cleaning — vectorised .map() |
| 6 | Categories | 01_data_cleaning — 10 mappings + subcategory |
| 7 | Delivery Days | 01_data_cleaning — range midpoint + median fill |
| 8 | Duplicates | data_cleaning — smart bulk aggregation |
| 9 | Price Outliers | data_cleaning_pipeline — cross-validation + Z-score |
| 10 | Payment Methods | 01_data_cleaning — 16 mappings + bar chart |

**Workflow:**  
- **Cell 2** → load one sample file  
- **Cells 3–12** → BEFORE / define function / AFTER on sample  
- **Cell 13** → silent batch pipeline over all files → save individual + merge master  
- **Cells 14–20** → feature engineering, insights, save  

---

## ① Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import re
import glob
import os
from datetime import datetime
from scipy.stats import zscore

In [2]:
# ── Path config — update these to match your folder layout ───────────────────
RAW_FOLDER     = '../data/raw/'
CLEANED_FOLDER = '../data/cleaned/'
MASTER_OUTPUT  = '../data/cleaned/amazon_india_master_cleaned.csv'
SAMPLE_FILE    = RAW_FOLDER + 'amazon_india_2025.csv'   # used for cells 3–12

os.makedirs(CLEANED_FOLDER, exist_ok=True)
print('✅ Libraries loaded.')
print(f'   Raw folder    : {RAW_FOLDER}')
print(f'   Cleaned folder: {CLEANED_FOLDER}')
print(f'   Sample file   : {SAMPLE_FILE}')

✅ Libraries loaded.
   Raw folder    : ../data/raw/
   Cleaned folder: ../data/cleaned/
   Sample file   : ../data/raw/amazon_india_2025.csv


## ② Load Sample File
> One file only — used to develop and verify all cleaning functions in cells 3–12.

In [3]:
df = pd.read_csv(SAMPLE_FILE)

print(f'✅ Loaded: {os.path.basename(SAMPLE_FILE)}')
print(f'   Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nColumn dtypes:')
print(df.dtypes.to_string())
print(f'\nFirst 3 rows:')
df.head(3)

✅ Loaded: amazon_india_2025.csv
   Shape  : 77,385 rows × 34 columns

Column dtypes:
transaction_id             object
order_date                 object
customer_id                object
product_id                 object
product_name               object
category                   object
subcategory                object
brand                      object
original_price_inr         object
discount_percent          float64
discounted_price_inr      float64
quantity                    int64
subtotal_inr              float64
delivery_charges          float64
final_amount_inr          float64
customer_city              object
customer_state             object
customer_tier              object
customer_spending_tier     object
customer_age_group         object
payment_method             object
delivery_days              object
delivery_type              object
is_prime_member            object
is_festival_sale           object
festival_name              object
customer_rating            obje

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2025_00000001,2025-01-08,CUST_2025_00005600,PROD_000627,Oppo F11 Pro 128GB Black,Electronics,Smartphones,Oppo,10234.12,0.00,...,False,NaN,4.5,Delivered,1,2025,1,0.15,True,4.4
1,TXN_2025_00000002,01/15/2025,CUST_2022_00027099,PROD_001699,Samsung Slate 4GB RAM Silver,Electronics,Tablets,Samsung,38241.08,0.00,...,False,NaN,NaN,Returned,1,2025,1,0.64,TRUE,3.4
2,TXN_2025_00000003,2025-01-26,CUST_2021_00027917,PROD_001242,Apple iPhone 16 Plus 64GB Black,Electronics,Smartphones,Apple,121974.26,32.04,...,True,Republic Day Sale,NaN,Returned,1,2025,1,0.18,True,3.4


## ③ Cleaning Functions
> Each cell: **BEFORE** (diagnose on `df`) → **define function** → **AFTER** (run on `df`)  
> All functions are reused silently in the batch pipeline (Cell 13).

### Cell 3 — Dates
> Source: `data_cleaning_pipeline` — 9-format regex map + unparsed leftover audit

In [4]:
# ── BEFORE ────────────────────────────────────────────────────────────────────
print('━'*55)
print('BEFORE — order_date')
print('━'*55)
print(f'dtype   : {df["order_date"].dtype}')
print(f'Missing : {df["order_date"].isna().sum():,}')
already_clean = df['order_date'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
print(f'Already YYYY-MM-DD : {already_clean.sum():,}')
print(f'Messy              : {(~already_clean).sum():,}')
print('Sample messy values:')
print(df.loc[~already_clean, 'order_date'].dropna().unique()[:12].tolist())

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE — order_date
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
dtype   : object
Missing : 0
Already YYYY-MM-DD : 69,961
Messy              : 7,424
Sample messy values:
['01/15/2025', '26-01-2025', '04-01-2025', '24-01-2025', '16/01/2025', '29-01-2025', '01/18/2025', '06/01/2025', '13/01/2025', '11-01-2025', '01/01/2025', '07/01/2025']


In [5]:
# ── FUNCTION ──────────────────────────────────────────────────────────────────
FORMAT_MAP = {
    r'^\d{4}-(0[1-9]|1[0-2])-(1[3-9]|[23]\d)$' : '%Y-%m-%d',
    r'^(1[3-9]|[23]\d)-(0[1-9]|1[0-2])-\d{4}$'  : '%d-%m-%Y',
    r'^(0[1-9]|1[0-2])-(1[3-9]|[23]\d)-\d{4}$'  : '%m-%d-%Y',
    r'^(1[3-9]|[23]\d)/(0[1-9]|1[0-2])/\d{4}$'  : '%d/%m/%Y',
    r'^(0[1-9]|1[0-2])/(1[3-9]|[23]\d)/\d{4}$'  : '%m/%d/%Y',
    r'^\d{4}-(1[3-9]|[23]\d)-(0[1-9]|1[0-2])$'  : '%Y-%d-%m',
    r'^\d{4}-\d{2}-\d{2}$'                        : '%Y-%m-%d',
    r'^\d{2}-\d{2}-\d{4}$'                        : '%d-%m-%Y',
    r'^\d{2}/\d{2}/\d{4}$'                        : '%d/%m/%Y',
}

def clean_dates(data):
    def _parse(val):
        if pd.isna(val) or not isinstance(val, str):
            return val
        val = val.strip()
        if re.match(r'(?i)^n/a$', val):
            return pd.NaT
        for pattern, fmt in FORMAT_MAP.items():
            if re.match(pattern, val):
                try:
                    return datetime.strptime(val, fmt).strftime('%Y-%m-%d')
                except ValueError:
                    return pd.NaT
        return val

    data['order_date'] = data['order_date'].astype(str).apply(_parse)
    return data

In [6]:
# ── AFTER ─────────────────────────────────────────────────────────────────────
df = clean_dates(df)
print()
print('━'*55)
print('AFTER — order_date')
print('━'*55)
clean_mask = df['order_date'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
print(f'Successfully formatted : {clean_mask.sum():,}')
print(f'Remaining unparsed     : {(~clean_mask).sum():,}')
print(f'Missing / NaT          : {df["order_date"].isna().sum():,}')
print('Leftover samples:', df.loc[~clean_mask, 'order_date'].head(5).tolist())


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER — order_date
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Successfully formatted : 77,385
Remaining unparsed     : 0
Missing / NaT          : 0
Leftover samples: []


### Cell 4 — Prices
> Source: NB1 diagnostics + NB3 regex map — strips ₹/Rs/commas, handles free/null/na

In [7]:
# ── BEFORE ────────────────────────────────────────────────────────────────────
print('━'*55)
print('BEFORE — original_price_inr / discounted_price_inr')
print('━'*55)
for col in ['original_price_inr', 'discounted_price_inr']:
    messy = df[col].astype(str).str.contains(r'[₹,a-zA-Z]', regex=True, na=False)
    print(f'{col}:')
    print(f'  Problematic entries : {messy.sum():,}')
    print(f'  Sample messy values : {df.loc[messy, col].unique()[:8].tolist()}')
    print(f'  Missing             : {df[col].isna().sum():,}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE — original_price_inr / discounted_price_inr
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
original_price_inr:
  Problematic entries : 6,934
  Sample messy values : ['Rs 49,127', 'Rs 21,896', '₹56,481.16', 'Rs 187,569', '₹42,906.33', 'Rs 20,928', '88,155.53', 'Rs 73,523']
  Missing             : 0
discounted_price_inr:
  Problematic entries : 0
  Sample messy values : []
  Missing             : 0


In [8]:
# ── FUNCTION ──────────────────────────────────────────────────────────────────
def clean_prices(data):
    PRICE_PATTERNS = [
        (r'^-?\d+(\.\d+)?$',                                             'float'),
        (r'(?i)^([₹$]|rs\.?|inr)\s*-?[\d,]+(\.\d+)?$',                'strip_symbol'),
        (r'^-?[\d,]+(\.\d+)?$',                                          'strip_comma'),
        (r'(?i)^(n/a|na|none|-|blank|null|missing|unknown|nan|inf)$',    'null'),
        (r'(?i)^free$',                                                   'zero'),
    ]

    def _parse(val):
        if pd.isna(val):
            return np.nan
        if isinstance(val, (int, float)):
            return float(val)
        val = str(val).strip()
        for pattern, action in PRICE_PATTERNS:
            if re.match(pattern, val):
                if action == 'float':        return float(val)
                if action == 'strip_symbol': return float(re.sub(r'[^\d.-]', '', val))
                if action == 'strip_comma':  return float(val.replace(',', ''))
                if action == 'null':         return np.nan
                if action == 'zero':         return 0.0
        return np.nan

    for col in ['original_price_inr', 'discounted_price_inr']:
        data[col] = data[col].apply(_parse)
        data[col] = pd.to_numeric(data[col], errors='coerce')
    return data

In [9]:
# ── AFTER ─────────────────────────────────────────────────────────────────────
df = clean_prices(df)
print()
print('━'*55)
print('AFTER — original_price_inr / discounted_price_inr')
print('━'*55)
for col in ['original_price_inr', 'discounted_price_inr']:
    valid = df[col].notna().sum()
    missing = df[col].isna().sum()
    print(f'{col}:')
    print(f'  Valid   : {valid:,}')
    print(f'  Missing : {missing:,}')
print()
print(df[['original_price_inr', 'discounted_price_inr']].describe().round(2))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER — original_price_inr / discounted_price_inr
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
original_price_inr:
  Valid   : 77,385
  Missing : 0
discounted_price_inr:
  Valid   : 77,385
  Missing : 0

       original_price_inr  discounted_price_inr
count            77385.00              77385.00
mean             60163.94              39822.21
std             311401.67              35258.99
min            -239246.47                405.53
25%              21351.47              16580.04
50%              31831.49              26527.34
75%              63597.67              52418.29
max           23751352.00             246106.63


### Cell 5 — Customer Ratings
> Source: `01_data_cleaning` — handles stars, fractions, range 1–5, median fill

In [10]:
# ── BEFORE ────────────────────────────────────────────────────────────────────
print('━'*55)
print('BEFORE — customer_rating')
print('━'*55)
print(f'dtype   : {df["customer_rating"].dtype}')
print(f'Unique  : {df["customer_rating"].unique()[:15].tolist()}')
print(f'Missing : {df["customer_rating"].isna().sum():,}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE — customer_rating
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
dtype   : object
Unique  : ['4.5', nan, '3.5', '5.0/5.0', '5.0', '3.0', '4.0', '4.0 stars', '3/5', '4.0/5.0', '3.5/5.0', '4.5 stars', '4.5/5.0', '4/5', '5.0 stars']
Missing : 23,464


In [11]:
# ── FUNCTION ──────────────────────────────────────────────────────────────────
def clean_ratings(data):
    def _parse(val):
        if pd.isna(val):
            return np.nan
        val = str(val).strip().lower()
        val = val.replace('stars', '').replace('star', '').strip()
        if '/' in val:
            try:
                n, d = val.split('/')
                result = float(n.strip()) / float(d.strip()) * 5.0
                return round(result, 1) if 1.0 <= result <= 5.0 else np.nan
            except (ValueError, ZeroDivisionError):
                return np.nan
        try:
            r = float(val)
            return round(r, 1) if 1.0 <= r <= 5.0 else np.nan
        except ValueError:
            return np.nan

    data['customer_rating'] = data['customer_rating'].apply(_parse)
    median_r = data['customer_rating'].median()
    data['customer_rating'] = data['customer_rating'].fillna(median_r)
    return data

In [12]:
# ── AFTER ─────────────────────────────────────────────────────────────────────
df = clean_ratings(df)
print()
print('━'*55)
print('AFTER — customer_rating')
print('━'*55)
print(f'Missing : {df["customer_rating"].isna().sum():,}')
print(f'Median used for fill: {df["customer_rating"].median()}')
print(df['customer_rating'].describe().round(2))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER — customer_rating
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Missing : 0
Median used for fill: 4.5
count    77385.00
mean         4.36
std          0.48
min          3.00
25%          4.00
50%          4.50
75%          4.50
max          5.00
Name: customer_rating, dtype: float64


### Cell 6 — Cities
> Source: `01_data_cleaning` — 13-entry map, trailing spaces, historical names

In [13]:
# ── BEFORE ────────────────────────────────────────────────────────────────────
print('━'*55)
print('BEFORE — customer_city')
print('━'*55)
print(f'Unique cities : {df["customer_city"].nunique()}')
print(sorted(df['customer_city'].dropna().unique().tolist()))

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE — customer_city
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Unique cities : 50
['Ahmedabad', 'Aligarh', 'Allahabad', 'BANGALORE', 'Bangalore', 'Banglore', 'Bareilly', 'Bengalore', 'Bengaluru', 'Bhubaneswar', 'Bombay', 'CHENNAI', 'Calcutta', 'Chandigarh', 'Chennai', 'Chennai ', 'Coimbatore', 'DELHI', 'Delhi', 'Delhi NCR', 'Gorakhpur', 'Hyderabad', 'Indore', 'Jaipur', 'KOLKATA', 'Kanpur', 'Kochi', 'Kolkata', 'Kolkata ', 'Lucknow', 'Ludhiana', 'MUMBAI', 'Madras', 'Meerut', 'Moradabad', 'Mumbai', 'Mumbai ', 'Nagpur', 'New Delhi', 'Patna', 'Pune', 'Saharanpur', 'Surat', 'Vadodara', 'Varanasi', 'Visakhapatnam', 'chenai', 'delhi', 'kolkata', 'mumba']


In [14]:
# ── FUNCTION ──────────────────────────────────────────────────────────────────
CITY_MAP = {
    'Bangalore'   : 'Bengaluru',
    'Banglore'    : 'Bengaluru',
    'Bengalore'   : 'Bengaluru',
    'Bombay'      : 'Mumbai',
    'Mumba'       : 'Mumbai',
    'Mumbai '     : 'Mumbai',
    'Delhi'       : 'New Delhi',
    'Delhi Ncr'   : 'New Delhi',
    'Calcutta'    : 'Kolkata',
    'Kolkata '    : 'Kolkata',
    'Madras'      : 'Chennai',
    'Chenai'      : 'Chennai',
    'Hydrabad'    : 'Hyderabad',
}

def clean_cities(data):
    data['customer_city'] = (
        data['customer_city']
        .astype(str).str.strip().str.title()
    )
    data['customer_city'] = data['customer_city'].replace(CITY_MAP)
    return data

In [15]:
# ── AFTER ─────────────────────────────────────────────────────────────────────
df = clean_cities(df)
print()
print('━'*55)
print('AFTER — customer_city')
print('━'*55)
print(f'Unique cities : {df["customer_city"].nunique()}')
print(sorted(df['customer_city'].dropna().unique().tolist()))
print()
print('Top 10 cities by orders:')
print(df['customer_city'].value_counts().head(10).to_string())


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER — customer_city
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Unique cities : 30
['Ahmedabad', 'Aligarh', 'Allahabad', 'Bareilly', 'Bengaluru', 'Bhubaneswar', 'Chandigarh', 'Chennai', 'Coimbatore', 'Gorakhpur', 'Hyderabad', 'Indore', 'Jaipur', 'Kanpur', 'Kochi', 'Kolkata', 'Lucknow', 'Ludhiana', 'Meerut', 'Moradabad', 'Mumbai', 'Nagpur', 'New Delhi', 'Patna', 'Pune', 'Saharanpur', 'Surat', 'Vadodara', 'Varanasi', 'Visakhapatnam']

Top 10 cities by orders:
customer_city
Mumbai       7264
New Delhi    6597
Bengaluru    5637
Pune         5220
Chennai      4621
Ahmedabad    3843
Kolkata      3669
Jaipur       3189
Surat        3027
Nagpur       3014


### Cell 7 — Booleans
> Source: `data_cleaning` — vectorised .map(), 8 variants, NaN → False

In [16]:
BOOL_COLS = ['is_prime_member', 'is_festival_sale', 'is_prime_eligible']

# ── BEFORE ────────────────────────────────────────────────────────────────────
print('━'*55)
print('BEFORE — boolean columns')
print('━'*55)
for col in BOOL_COLS:
    if col in df.columns:
        print(f'{col}: {df[col].unique().tolist()}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE — boolean columns
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
is_prime_member: ['False', 'True', '1', 'TRUE', 'No', 'FALSE', '0', 'Yes']
is_festival_sale: ['False', 'True', 'FALSE', 'No', '0', 'TRUE', 'Yes', '1']
is_prime_eligible: ['True', 'TRUE', 'False', '1', 'Yes', '0', 'FALSE', 'No']


In [17]:
BOOL_COLS = ['is_prime_member', 'is_festival_sale', 'is_prime_eligible']

# ── FUNCTION ──────────────────────────────────────────────────────────────────
BOOL_MAP = {
    'true' : True,  'yes' : True,  '1' : True,  'y' : True,
    'false': False, 'no'  : False, '0' : False, 'n' : False,
}

def clean_booleans(data):
    for col in BOOL_COLS:
        if col in data.columns:
            data[col] = (
                data[col]
                .astype(str).str.strip().str.lower()
                .map(BOOL_MAP)
                .fillna(False)
            )
    return data

In [18]:
# ── AFTER ─────────────────────────────────────────────────────────────────────
df = clean_booleans(df)
print()
print('━'*55)
print('AFTER — boolean columns')
print('━'*55)
for col in BOOL_COLS:
    if col in df.columns:
        true_n  = df[col].sum()
        false_n = (~df[col]).sum()
        pct     = df[col].mean() * 100
        print(f'{col}:')
        print(f'  True : {true_n:,}   False: {false_n:,}   True%: {pct:.1f}%')


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER — boolean columns
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
is_prime_member:
  True : 49,800   False: 27,585   True%: 64.4%
is_festival_sale:
  True : 23,635   False: 53,750   True%: 30.5%
is_prime_eligible:
  True : 64,033   False: 13,352   True%: 82.7%


### Cell 8 — Categories
> Source: `01_data_cleaning` — 10-entry map, Fashion/Books/Sports/Health, + subcategory clean

In [19]:
# ── BEFORE ────────────────────────────────────────────────────────────────────
print('━'*55)
print('BEFORE — category')
print('━'*55)
print(f'Unique categories : {df["category"].nunique()}')
for cat in sorted(df['category'].dropna().unique()):
    cnt = (df['category'] == cat).sum()
    print(f"  '{cat}' → {cnt:,}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE — category
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Unique categories : 5
  'ELECTRONICS' → 22
  'Electronic' → 15
  'Electronics' → 77,303
  'Electronics & Accessories' → 20
  'Electronicss' → 25


In [20]:
# ── FUNCTION ──────────────────────────────────────────────────────────────────
CAT_MAP = {
    'Electronic'                  : 'Electronics',
    'Electronicss'                : 'Electronics',
    'Electronics & Accessories'   : 'Electronics',
    'Clothing'                    : 'Fashion',
    'Clothes'                     : 'Fashion',
    'Apparel'                     : 'Fashion',
    'Books & Media'               : 'Books',
    'Sports & Outdoors'           : 'Sports',
    'Health & Beauty'             : 'Health & Personal Care',
    'Home And Kitchen'            : 'Home & Kitchen',
}

def clean_categories(data):
    data['category'] = (
        data['category']
        .astype(str).str.strip().str.title()
    )
    data['category'] = data['category'].replace(CAT_MAP)
    if 'subcategory' in data.columns:
        data['subcategory'] = (
            data['subcategory']
            .astype(str).str.strip().str.title()
        )
    return data

In [21]:
# ── AFTER ─────────────────────────────────────────────────────────────────────
df = clean_categories(df)
print()
print('━'*55)
print('AFTER — category')
print('━'*55)
print(f'Unique categories : {df["category"].nunique()}')
print()
for cat, cnt in df['category'].value_counts().items():
    pct = cnt / len(df) * 100
    print(f'  {cat:<30} {cnt:>8,}  ({pct:.1f}%)')


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER — category
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Unique categories : 1

  Electronics                      77,385  (100.0%)


### Cell 9 — Delivery Days
> Source: `01_data_cleaning` — range midpoint, 0–30 validation, median fill, performance stats

In [22]:
# ── BEFORE ────────────────────────────────────────────────────────────────────
print('━'*55)
print('BEFORE — delivery_days')
print('━'*55)
print(f'dtype   : {df["delivery_days"].dtype}')
print(f'Missing : {df["delivery_days"].isna().sum():,}')
print(f'Unique  : {sorted(df["delivery_days"].dropna().unique().tolist())[:20]}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE — delivery_days
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
dtype   : object
Missing : 0
Unique  : ['-1', '0', '1', '1-2 days', '15', '2', '3', '4', '5', '6', '7', 'Express', 'Same Day']


In [23]:
# ── FUNCTION ──────────────────────────────────────────────────────────────────
def clean_delivery_days(data):
    def _parse(val):
        if pd.isna(val):
            return np.nan
        val = str(val).strip().lower()
        if val in ['same day', 'same-day', 'sameday']:
            return 0.0
        if val in ['express', 'next day', 'next-day']:
            return 1.0
        val = val.replace('days', '').replace('day', '').strip()
        if '-' in val and not val.startswith('-'):
            try:
                lo, hi = val.split('-')
                return round((float(lo.strip()) + float(hi.strip())) / 2, 1)
            except ValueError:
                return np.nan
        try:
            d = float(val)
            return d if 0.0 <= d <= 30.0 else np.nan
        except ValueError:
            return np.nan

    data['delivery_days'] = data['delivery_days'].apply(_parse)
    median_d = data['delivery_days'].median()
    data['delivery_days'] = data['delivery_days'].fillna(median_d)
    return data

In [24]:
# ── AFTER ─────────────────────────────────────────────────────────────────────
df = clean_delivery_days(df)
print()
print('━'*55)
print('AFTER — delivery_days')
print('━'*55)
print(f'Missing : {df["delivery_days"].isna().sum():,}')
print(f'⚡ Same Day  (0d) : {(df["delivery_days"] == 0).sum():>8,}')
print(f'🚀 Next Day  (1d) : {(df["delivery_days"] == 1).sum():>8,}')
print(f'✅ Within 3d      : {(df["delivery_days"] <= 3).sum():>8,}')
print(f'⏱  Avg delivery   : {df["delivery_days"].mean():.1f} days')
print(f'📦 Max delivery   : {df["delivery_days"].max():.0f} days')


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER — delivery_days
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Missing : 0
⚡ Same Day  (0d) :      465
🚀 Next Day  (1d) :   24,538
✅ Within 3d      :   58,369
⏱  Avg delivery   : 2.6 days
📦 Max delivery   : 15 days


### Cell 10 — Duplicates
> Source: `data_cleaning` — removes _DUP system errors, aggregates genuine bulk orders

In [25]:
# ── BEFORE ────────────────────────────────────────────────────────────────────
print('━'*55)
print('BEFORE — duplicates')
print('━'*55)
print(f'Total rows          : {len(df):,}')
print(f'Exact duplicates    : {df.duplicated().sum():,}')
if 'transaction_id' in df.columns:
    dup_ids = df[df['transaction_id'].astype(str).str.endswith('_DUP')]
    print(f'_DUP system errors  : {len(dup_ids):,}')
    bulk = df[df.duplicated(subset=['transaction_id'], keep=False) &
              ~df['transaction_id'].astype(str).str.endswith('_DUP')]
    print(f'Genuine bulk orders : {len(bulk):,}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE — duplicates
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total rows          : 77,385
Exact duplicates    : 0
_DUP system errors  : 385
Genuine bulk orders : 0


In [26]:
# ── FUNCTION ──────────────────────────────────────────────────────────────────
def handle_duplicates(data):
    # Step 1 — remove system-generated _DUP entries
    if 'transaction_id' in data.columns:
        dup_mask = data['transaction_id'].astype(str).str.endswith('_DUP')
        data = data[~dup_mask].copy()

    # Step 2 — aggregate genuine bulk orders
    if 'transaction_id' in data.columns:
        agg_cols = {}
        for col in data.columns:
            if col == 'transaction_id':
                continue
            elif col in ['quantity', 'order_subtotal_inr', 'final_amount_inr',
                         'discount_amount_inr']:
                agg_cols[col] = 'sum'
            else:
                agg_cols[col] = 'first'
        data = data.groupby('transaction_id', as_index=False).agg(agg_cols)
    else:
        # Fallback: drop exact duplicates
        data = data.drop_duplicates()

    return data

In [27]:
# ── AFTER ─────────────────────────────────────────────────────────────────────
df = handle_duplicates(df)
print()
print('━'*55)
print('AFTER — duplicates')
print('━'*55)
print(f'Total rows remaining : {len(df):,}')
print(f'Exact duplicates     : {df.duplicated().sum():,}')


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER — duplicates
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total rows remaining : 77,000
Exact duplicates     : 0


### Cell 11 — Price Outliers
> Source: `data_cleaning_pipeline` — cross-validation vs derived price + Z-score ratio fix

In [28]:
# ── BEFORE — cross-validation ─────────────────────────────────────────────────
print('━'*55)
print('BEFORE — price outlier cross-validation')
print('━'*55)
if 'discount_percent' in df.columns:
    disc_pct = pd.to_numeric(df['discount_percent'], errors='coerce') / 100
    df['_derived_price'] = (df['discounted_price_inr'] / (1 - disc_pct)).round(2)
    df['_price_diff']    = (df['original_price_inr'] - df['_derived_price']).abs()
    mismatch = df[df['_price_diff'] > 100]
    print(f'Rows where |original - derived| > ₹100 : {len(mismatch):,}')
    if len(mismatch) > 0:
        print(mismatch[['original_price_inr', '_derived_price', '_price_diff']].head(10).to_string())
else:
    print('(discount_percent column not found — skipping cross-validation preview)')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE — price outlier cross-validation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Rows where |original - derived| > ₹100 : 562
     original_price_inr  _derived_price  _price_diff
68           3183538.00        31835.38   3151702.62
164          -128145.60       128139.44    256285.04
405           324215.00        32423.00    291792.00
497           -13158.23        13158.23     26316.46
577           -24345.39        24346.07     48691.46
658          3138383.00        31383.83   3106999.17
724          6966899.00        69668.99   6897230.01
734          1075470.50       107547.05    967923.45
799          2485603.00        24856.03   2460746.97
806          7448213.00        74482.13   7373730.87


In [29]:
# ── FUNCTION ──────────────────────────────────────────────────────────────────
def fix_price_outliers(data):
    # Fix negatives
    data['original_price_inr'] = data['original_price_inr'].abs()

    # Cross-validation: derive expected price from discount%
    if 'discount_percent' in data.columns:
        disc_pct = pd.to_numeric(data['discount_percent'], errors='coerce') / 100
        data['_derived_price'] = (data['discounted_price_inr'] / (1 - disc_pct)).round(2)
        data['_price_diff']    = (data['original_price_inr'] - data['_derived_price']).abs()

    # Z-score on price ratio
    safe_disc = data['discounted_price_inr'].replace(0, np.nan)
    data['_ratio'] = data['original_price_inr'] / safe_disc
    data['_zscore'] = zscore(data['_ratio'].fillna(data['_ratio'].median()))

    mask_100x = (data['_zscore'] > 3) & (data['_ratio'] > 50)
    mask_10x  = (data['_zscore'] > 3) & (data['_ratio'] <= 50)
    data.loc[mask_100x, 'original_price_inr'] = \
        data.loc[mask_100x, 'original_price_inr'] / 100
    data.loc[mask_10x,  'original_price_inr'] = \
        data.loc[mask_10x,  'original_price_inr'] / 10

    drop_cols = [c for c in ['_derived_price', '_price_diff', '_ratio', '_zscore']
                 if c in data.columns]
    return data.drop(columns=drop_cols)

In [30]:
# ── AFTER — re-check mismatches ───────────────────────────────────────────────
df = fix_price_outliers(df)
print()
print('━'*55)
print('AFTER — price outlier cross-validation')
print('━'*55)
if 'discount_percent' in df.columns:
    disc_pct2 = pd.to_numeric(df['discount_percent'], errors='coerce') / 100
    derived2  = (df['discounted_price_inr'] / (1 - disc_pct2)).round(2)
    diff2     = (df['original_price_inr'] - derived2).abs()
    remaining = (diff2 > 100).sum()
    print(f'Remaining mismatches > ₹100 : {remaining:,}')
print(df[['original_price_inr', 'discounted_price_inr']].describe().round(2))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER — price outlier cross-validation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Remaining mismatches > ₹100 : 162
       original_price_inr  discounted_price_inr
count            77000.00              77000.00
mean             49077.71              39815.77
std              48708.31              35270.52
min               1067.27                405.53
25%              21421.50              16569.19
50%              31831.49              26527.34
75%              63072.61              52411.37
max            2316168.40             246106.63


### Cell 12 — Payment Methods
> Source: `01_data_cleaning` — 16-entry map incl. Paytm/BHIM/BNPL/Wallet + terminal bar chart

In [31]:
# ── BEFORE ────────────────────────────────────────────────────────────────────
print('━'*55)
print('BEFORE — payment_method')
print('━'*55)
print(f'Unique : {df["payment_method"].nunique()}')
for m, cnt in df['payment_method'].value_counts().items():
    print(f"  '{m}' → {cnt:,}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE — payment_method
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Unique : 7
  'UPI' → 46,323
  'Credit Card' → 9,227
  'COD' → 6,207
  'Debit Card' → 6,197
  'BNPL' → 5,238
  'Net Banking' → 2,270
  'Wallet' → 1,538


In [32]:
# ── FUNCTION ──────────────────────────────────────────────────────────────────
METHOD_MAP = {
    'Cod'            : 'Cash on Delivery',
    'C.O.D'          : 'Cash on Delivery',
    'Cash On Delivery': 'Cash on Delivery',
    'Upi'            : 'UPI',
    'Phonepe'        : 'UPI',
    'Googlepay'      : 'UPI',
    'Gpay'           : 'UPI',
    'Paytm'          : 'UPI',
    'Bhim'           : 'UPI',
    'Credit_Card'    : 'Credit Card',
    'Cc'             : 'Credit Card',
    'Debit_Card'     : 'Debit Card',
    'Dc'             : 'Debit Card',
    'Net_Banking'    : 'Net Banking',
    'Netbanking'     : 'Net Banking',
    'Bnpl'           : 'BNPL',
    'Snapmint'       : 'BNPL',
    'Mobikwik'       : 'Wallet',
    'Freecharge'     : 'Wallet',
}

def clean_payment_methods(data):
    data['payment_method'] = (
        data['payment_method']
        .astype(str).str.strip().str.title()
    )
    data['payment_method'] = data['payment_method'].replace(METHOD_MAP)
    return data

In [33]:
# ── AFTER — bar chart ─────────────────────────────────────────────────────────
df = clean_payment_methods(df)
print()
print('━'*55)
print('AFTER — payment_method')
print('━'*55)
print(f'Unique : {df["payment_method"].nunique()}')
print()
for m, cnt in df['payment_method'].value_counts().items():
    bar = '█' * int(cnt / len(df) * 40)
    print(f'  {m:<22} {cnt:>8,}  {bar}')


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER — payment_method
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Unique : 7

  UPI                      46,323  ████████████████████████
  Credit Card               9,227  ████
  Cash on Delivery          6,207  ███
  Debit Card                6,197  ███
  BNPL                      5,238  ██
  Net Banking               2,270  █
  Wallet                    1,538  


In [34]:
def master_pipeline(data):
    """Apply all 10 cleaning steps in order."""
    data = clean_dates(data)
    data = clean_prices(data)
    data = clean_ratings(data)
    data = clean_cities(data)
    data = clean_booleans(data)
    data = clean_categories(data)
    data = clean_delivery_days(data)
    data = handle_duplicates(data)
    data = fix_price_outliers(data)
    data = clean_payment_methods(data)
    return data

## ④ Batch Pipeline — All Files
> Functions from cells 3–12 are applied silently to every yearly CSV.  
> Each file is saved individually, then all are merged into one master file.

In [35]:


all_files   = sorted(glob.glob(RAW_FOLDER + 'amazon_india_*.csv'))
master_list = []

print(f'Processing {len(all_files)} files...\n')

for filepath in all_files:
    fname = os.path.basename(filepath)
    year  = fname.replace('amazon_india_', '').replace('.csv', '')

    raw   = pd.read_csv(filepath)
    clean = master_pipeline(raw)

    out_path = CLEANED_FOLDER + f'clean_amazon_india_{year}.csv'
    clean.to_csv(out_path, index=False)
    master_list.append(clean)

    print(f'  ✅ {fname:<35} {len(raw):>8,} → {len(clean):>8,} rows  →  saved')

# Merge all into master
master_df = pd.concat(master_list, ignore_index=True)
print('━'*55)
print(f'Master df shape : {master_df.shape[0]:,} rows × {master_df.shape[1]} columns')
print(f'Years covered   : {sorted(master_df["order_date"].astype(str).str[:4].unique().tolist())}')

Processing 11 files...

  ✅ amazon_india_2015.csv                 33,165 →   33,000 rows  →  saved
  ✅ amazon_india_2016.csv                 55,275 →   55,000 rows  →  saved
  ✅ amazon_india_2017.csv                 77,385 →   77,000 rows  →  saved
  ✅ amazon_india_2018.csv                 99,495 →   99,000 rows  →  saved
  ✅ amazon_india_2019.csv                121,605 →  121,000 rows  →  saved
  ✅ amazon_india_2020.csv                143,715 →  143,000 rows  →  saved
  ✅ amazon_india_2021.csv                138,187 →  137,500 rows  →  saved
  ✅ amazon_india_2022.csv                132,660 →  132,000 rows  →  saved
  ✅ amazon_india_2023.csv                127,132 →  126,500 rows  →  saved
  ✅ amazon_india_2024.csv                121,605 →  121,000 rows  →  saved
  ✅ amazon_india_2025.csv                 77,385 →   77,000 rows  →  saved
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Master df shape : 1,122,000 rows × 34 columns
Years covered   : ['2015', '2016', '2017', '2018'

This one for saving the Master df into CSV

In [36]:
# ── Save master CSV ───────────────────────────────────────────────────────────
master_df.to_csv(MASTER_OUTPUT, index=False)

print('━'*60)
print('✅ MASTER FILE SAVED')
print('━'*60)
print(f'  Path    : {MASTER_OUTPUT}')
print(f'  Rows    : {len(master_df):,}')
print(f'  Columns : {master_df.shape[1]}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ MASTER FILE SAVED
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Path    : ../data/cleaned/amazon_india_master_cleaned.csv
  Rows    : 1,122,000
  Columns : 34


## ⑤ Feature Engineering
> 11 derived columns added to master_df

In [37]:
# dates = pd.to_datetime(master_df['order_date'], errors='coerce')

# # 1. order_day — day of month
# master_df['order_day']         = dates.dt.day

# # 2. order_day_of_week — Mon=0, Sun=6
# master_df['order_day_of_week'] = dates.dt.dayofweek

# # 3. is_weekend — Sat or Sun
# master_df['is_weekend']        = master_df['order_day_of_week'].isin([5, 6])

# # 4. discount_amount_inr
# master_df['discount_amount_inr'] = (
#     master_df['original_price_inr'] - master_df['discounted_price_inr']
# ).clip(lower=0)

# # 5. price_per_unit
# qty = master_df['quantity'] if 'quantity' in master_df.columns else 1
# master_df['price_per_unit'] = (master_df['discounted_price_inr'] / qty).round(2)

# # 6. is_free_delivery
# master_df['is_free_delivery'] = master_df['delivery_days'] == 0

# # 7. is_returned
# if 'return_status' in master_df.columns:
#     master_df['is_returned'] = master_df['return_status'].astype(str).str.lower().isin(
#         ['returned', 'return', 'yes', '1', 'true'])
# else:
#     master_df['is_returned'] = False

# # 8. is_fast_delivery — delivered within 2 days
# master_df['is_fast_delivery'] = master_df['delivery_days'] <= 2

# # 9. revenue_band
# master_df['revenue_band'] = pd.cut(
#     master_df['original_price_inr'],
#     bins   = [0, 500, 2000, 10000, np.inf],
#     labels = ['Budget', 'Mid-Range', 'Premium', 'Luxury'],
#     right  = True
# )

# # 10. discount_band
# disc_pct = pd.to_numeric(master_df.get('discount_percent', pd.Series(0)), errors='coerce')
# master_df['discount_band'] = pd.cut(
#     disc_pct,
#     bins   = [-1, 10, 25, 50, 100],
#     labels = ['Low (0-10%)', 'Medium (11-25%)', 'High (26-50%)', 'Very High (51%+)']
# )

# # 11. estimated_profit_inr (10% margin assumption)
# master_df['estimated_profit_inr'] = (master_df['discounted_price_inr'] * 0.10).round(2)

# print('✅ Feature engineering complete.')
# print(f'   Columns now : {master_df.shape[1]}')
# print('   New columns  :', ['order_day','order_day_of_week','is_weekend','discount_amount_inr',
#                              'price_per_unit','is_free_delivery','is_returned','is_fast_delivery',
#                              'revenue_band','discount_band','estimated_profit_inr'])

## ⑥ Business Insights

In [38]:
# # ── Dataset overview ──────────────────────────────────────────────────────────
# print('━'*55)
# print('📦 DATASET OVERVIEW')
# print('━'*55)
# print(f'Total rows    : {len(master_df):,}')
# print(f'Total columns : {master_df.shape[1]}')
# years = master_df['order_date'].astype(str).str[:4]
# print(f'Year range    : {years.min()} → {years.max()}')
# print(f'Years covered : {sorted(years.unique().tolist())}')
# print(f'Missing values: {master_df.isnull().sum().sum():,} total across all columns')

In [39]:
# # ── Prime & festival stats ────────────────────────────────────────────────────
# print('━'*55)
# print('👑 PRIME & FESTIVAL STATS')
# print('━'*55)
# if 'is_prime_member' in master_df.columns:
#     pct = master_df['is_prime_member'].mean() * 100
#     print(f'Prime members     : {master_df["is_prime_member"].sum():>10,}  ({pct:.1f}%)')
# if 'is_prime_eligible' in master_df.columns:
#     pct = master_df['is_prime_eligible'].mean() * 100
#     print(f'Prime eligible    : {master_df["is_prime_eligible"].sum():>10,}  ({pct:.1f}%)')
# if 'is_festival_sale' in master_df.columns:
#     pct = master_df['is_festival_sale'].mean() * 100
#     print(f'Festival orders   : {master_df["is_festival_sale"].sum():>10,}  ({pct:.1f}%)')
# if 'is_weekend' in master_df.columns:
#     pct = master_df['is_weekend'].mean() * 100
#     print(f'Weekend orders    : {master_df["is_weekend"].sum():>10,}  ({pct:.1f}%)')

In [40]:
# # ── Delivery performance ──────────────────────────────────────────────────────
# print('━'*55)
# print('🚚 DELIVERY PERFORMANCE')
# print('━'*55)
# print(f'⚡ Same Day   (0d)  : {(master_df["delivery_days"] == 0).sum():>10,}')
# print(f'🚀 Next Day   (1d)  : {(master_df["delivery_days"] == 1).sum():>10,}')
# print(f'✅ Within 2d        : {(master_df["delivery_days"] <= 2).sum():>10,}')
# print(f'📦 Within 3d        : {(master_df["delivery_days"] <= 3).sum():>10,}')
# print(f'⏱  Avg delivery     : {master_df["delivery_days"].mean():.2f} days')
# print(f'📏 Max delivery     : {master_df["delivery_days"].max():.0f} days')
# print(f'🚄 Fast delivery %  : {master_df["is_fast_delivery"].mean()*100:.1f}%')

In [41]:
# # ── Top cities + payment evolution ────────────────────────────────────────────
# print('━'*55)
# print('🏙️  TOP 10 CITIES BY ORDERS')
# print('━'*55)
# city_counts = master_df['customer_city'].value_counts().head(10)
# for city, cnt in city_counts.items():
#     bar = '█' * int(cnt / city_counts.max() * 30)
#     pct = cnt / len(master_df) * 100
#     print(f'  {city:<18} {cnt:>8,}  ({pct:.1f}%)  {bar}')

# print()
# print('━'*55)
# print('💳 PAYMENT METHOD EVOLUTION BY YEAR')
# print('━'*55)
# master_df['_year'] = master_df['order_date'].astype(str).str[:4]
# pivot = pd.crosstab(master_df['_year'], master_df['payment_method'])
# pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).mul(100).round(1)
# print(pivot_pct.to_string())
# master_df.drop(columns=['_year'], inplace=True, errors='ignore')

In [42]:
# # ── Price & revenue band breakdown ────────────────────────────────────────────
# print('━'*55)
# print('💰 PRICE BY SUBCATEGORY (top 15)')
# print('━'*55)
# if 'subcategory' in master_df.columns:
#     price_stats = (
#         master_df.groupby('subcategory')['original_price_inr']
#         .agg(['mean', 'min', 'max', 'count'])
#         .sort_values('mean', ascending=False)
#         .head(15)
#         .round(0)
#     )
#     price_stats.columns = ['Avg ₹', 'Min ₹', 'Max ₹', 'Count']
#     print(price_stats.to_string())

# print()
# print('━'*55)
# print('🏷️  REVENUE BAND DISTRIBUTION')
# print('━'*55)
# for band, cnt in master_df['revenue_band'].value_counts().sort_index().items():
#     pct = cnt / len(master_df) * 100
#     bar = '█' * int(pct / 2)
#     print(f'  {str(band):<18} {cnt:>8,}  ({pct:.1f}%)  {bar}')

## ⑦ Save Output + Cleaning Summary

In [43]:
# # ── Save master CSV ───────────────────────────────────────────────────────────
# master_df.to_csv(MASTER_OUTPUT, index=False)

# print('━'*60)
# print('✅ MASTER FILE SAVED')
# print('━'*60)
# print(f'  Path    : {MASTER_OUTPUT}')
# print(f'  Rows    : {len(master_df):,}')
# print(f'  Columns : {master_df.shape[1]}')

# print()
# print('━'*60)
# print('🧹 10-STEP CLEANING SUMMARY')
# print('━'*60)
# steps = [
#     ('1  Dates',            'data_cleaning_pipeline', '9-format regex → standardised YYYY-MM-DD'),
#     ('2  Prices',           'NB1 + NB3 combined',     'Stripped ₹/Rs/commas, nulls, free → 0'),
#     ('3  Ratings',          '01_data_cleaning',       'Fractions, stars → float; out-of-range → median'),
#     ('4  Cities',           '01_data_cleaning',       '13-entry map; historical names + typos fixed'),
#     ('5  Booleans',         'data_cleaning',          'Vectorised map; yes/no/1/0/y/n → True/False'),
#     ('6  Categories',       '01_data_cleaning',       '10-entry map; Fashion/Books/Sports added; subcategory cleaned'),
#     ('7  Delivery Days',    '01_data_cleaning',       'Ranges → midpoint; 0–30 validated; missing → median'),
#     ('8  Duplicates',       'data_cleaning',          '_DUP system errors removed; bulk orders aggregated'),
#     ('9  Price Outliers',   'data_cleaning_pipeline', 'Cross-validation vs derived price; Z-score ÷10/÷100 fix'),
#     ('10 Payment Methods',  '01_data_cleaning',       '16-entry map; Paytm/BHIM/BNPL/Wallet consolidated'),
# ]
# for step, source, detail in steps:
#     print(f'  Step {step:<20} [{source:<25}]  {detail}')

# print()
# print('  + 11 feature engineering columns added')
# print('  + Business insights printed in cells 15–19')
# print()
# print('🎉 Pipeline complete!')